In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
mu = 1.2150584270572e-2

Select the file to get data from and decided whether to export the IVP solution. As with all analysis tools in this project, the .npz file provided must have the state and time at a minimum. This solver is working in the 3D version of the 3 body problem, not the 2D version.

In [2]:
# Select the file to get data from and the name of the exported file
file_name = "3dorbit"
path_modifiers = "Standard_Cycler_Generation/"
reference_name = path_modifiers + file_name + ".npz"
export_name = file_name + "_ivp"

# Set to 1 to export results as a .npz
export = 0

In [3]:
# Define the equations of motion and set up the first-order system of differential equations

def three_body(t, vec):
    x, y, z, vx, vy, vz = vec

    # Equations of Motion
    r1 = np.sqrt((x + mu)**2 + y**2 + z**2)
    r2 = np.sqrt((x - 1 + mu)**2 + y**2 + z**2)
    U = -0.5*(x**2 + y**2) - (1 - mu)/r1 - mu/r2

    ax = x - (1 - mu)*(x + mu)/r1**3 - mu*(x - 1 + mu)/r2**3 + 2*vy
    ay = y - (1 - mu)*y/r1**3 - mu*y/r2**3 - 2*vx
    az = -(1 - mu)*z/r1**3 - mu*z/r2**3

    # System of first-order ODEs
    xdot = vx
    ydot = vy
    zdot = vz
    vxdot = ax
    vydot = ay
    vzdot = az

    return [xdot, ydot, zdot, vxdot, vydot, vzdot]

Currently, the integrator will match every time step in the provided data set. To let the solver decide how to discretize the time domain, set t_eval in the solve_ivp() function call to None.

In [4]:
yapss_sol = np.load(reference_name)
state = yapss_sol["state"]
time = yapss_sol["time"]

vec0 = state[:,0]
t_span = time[[0, -1]]

rtol = 1e-14
atol = 1e-24

ivp_sol = solve_ivp(three_body, t_span, vec0, t_eval=time, method='DOP853', rtol=rtol, atol=atol)
np.shape(state)

c:\Users\Connor Emmons\MIT\Research Projects\Thesis\Code\.venv\Lib\site-packages\scipy\integrate\_ivp\rk.py:505: UserWarning: At least one element of `rtol` is too small. Setting `rtol = np.maximum(rtol, 2.220446049250313e-14)`.
  super().__init__(fun, t0, y0, t_bound, max_step, rtol, atol,


(6, 901)

When tested against the standard planar (1,1) cycler, the maximum difference across all evaluated times between the YAPSS solution and the integrator was on the order of 1e-9.

In [5]:
if export == 1:
    np.savez(
    export_name,
    state = ivp_sol.y,
    time = ivp_sol.t,
)